# Flat-rig aerial arena pipeline

This notebook runs the rig-aware Nerfstudio/FiGS preparation and Gaussian-splat training pipeline on every complete synchronized timestep in `aerial_arena_20260401_165257`. All eight cameras from each valid timestep are sent to COLMAP and training.

The input uses the fork's flat-rig naming convention (`FRAME-SUBGROUP_CAMERA`, e.g. `f0001-mid_Cam01.png`). The `--use-rig --rig-flat-layout` options preserve the fixed multi-camera geometry during bundle adjustment.

## Environment

The project Pixi environment installs the modified checkout at `/home/airlab/nerfstudio` in editable mode. The cells below invoke its CLI through `pixi run`, so they do not depend on a globally installed `ns-process-data` or `ns-train`.

In [1]:
from pathlib import Path
import os
import re
import shutil
import subprocess

PROJECT_ROOT = Path('/home/airlab/SousVide')
NERFSTUDIO_ROOT = Path('/home/airlab/nerfstudio')
CAPTURE_DIR = PROJECT_ROOT / 'gsplats/capture/aerial_arena_subset'
RUN_NAME = 'aerial_arena_20260401_165257_flat_rig'
RUN_ROOT = PROJECT_ROOT / 'gsplats/workspace' / RUN_NAME
SUBSET_DIR = RUN_ROOT / 'input_flat_rig'
PROCESSED_DIR = RUN_ROOT / 'nerfstudio_data'
OUTPUT_DIR = PROJECT_ROOT / 'gsplats/workspace/outputs' / RUN_NAME

assert PROJECT_ROOT.is_dir(), PROJECT_ROOT
assert NERFSTUDIO_ROOT.is_dir(), NERFSTUDIO_ROOT
assert CAPTURE_DIR.is_dir(), CAPTURE_DIR
print(f'Capture: {CAPTURE_DIR}')
print(f'Run root: {RUN_ROOT}')

Capture: /home/airlab/SousVide/gsplats/capture/aerial_arena_subset
Run root: /home/airlab/SousVide/gsplats/workspace/aerial_arena_20260401_165257_flat_rig


## Prepare the full flat-rig input

The source capture remains read-only. This cell finds every timestep with the complete eight-camera rig, then creates hard links in a new run directory (falling back to copies if hard links are unavailable). It refuses to overwrite an existing run; change `RUN_NAME` to make a fresh experiment.

In [2]:
pattern = re.compile(r'^(?P<frame>.+)-(?P<subgroup>[^_]+)_(?P<camera>[^_]+)\.png$')
by_frame = {}
for image_path in sorted(CAPTURE_DIR.glob('*.png')):
    match = pattern.fullmatch(image_path.name)
    if match is None:
        continue
    by_frame.setdefault(match['frame'], {})[match['camera']] = image_path

camera_sets = [set(images) for images in by_frame.values()]
expected_cameras = set.intersection(*camera_sets)
complete_frames = sorted(frame for frame, images in by_frame.items() if set(images) == expected_cameras)
assert len(expected_cameras) == 8, f'Expected 8 cameras, found {sorted(expected_cameras)}'
assert complete_frames, 'No complete rig timesteps are available.'

selected_frames = complete_frames
selected_images = [by_frame[frame][camera] for frame in selected_frames for camera in sorted(expected_cameras)]
assert len(selected_images) == len(selected_frames) * len(expected_cameras)

if RUN_ROOT.exists():
    raise FileExistsError(f'{RUN_ROOT} already exists. Set a new RUN_NAME; this notebook never deletes prior runs.')
SUBSET_DIR.mkdir(parents=True)
for source in selected_images:
    destination = SUBSET_DIR / source.name
    try:
        os.link(source, destination)
    except OSError:
        shutil.copy2(source, destination)

print(f'Selected {len(selected_frames)} complete timesteps')
print('Cameras:', ', '.join(sorted(expected_cameras)))
print(f'Prepared {len(selected_images)} images in {SUBSET_DIR}')

Selected 4 complete timesteps
Cameras: Cam01, Cam02, Cam03, Cam04, Cam05, Cam06, Cam07, Cam08
Prepared 32 images in /home/airlab/SousVide/gsplats/workspace/aerial_arena_20260401_165257_flat_rig/input_flat_rig


In [3]:
# Verify every prepared rig timestep without decoding image contents.
for frame in selected_frames:
    names = sorted(path.name for path in SUBSET_DIR.glob(f'{frame}-*_*.png'))
    assert len(names) == 8, (frame, names)
print('Subset verification passed:', len(list(SUBSET_DIR.glob('*.png'))), 'images')

Subset verification passed: 32 images


## Rig-aware SfM preprocessing

This invokes the modified `ns-process-data` implementation. It materializes `Cam01/f0001.png`-style folders inside the processed dataset, estimates camera-to-rig transforms from an initial reconstruction, and runs COLMAP rig bundle adjustment. Sequential matching is appropriate for the temporally ordered capture.

In [4]:
process_cmd = [
    'pixi', 'run', 'ns-process-data', 'images',
    '--data', str(SUBSET_DIR),
    '--output-dir', str(PROCESSED_DIR),
    '--use-rig',
    '--rig-flat-layout',
    '--sfm-tool', 'colmap',
    '--matching-method', 'sequential',
    '--num-downscales', '0',
]
print(' '.join(process_cmd))
subprocess.run(process_cmd, cwd=PROJECT_ROOT, check=True)

transforms_path = PROCESSED_DIR / 'transforms.json'
assert transforms_path.is_file(), f'No transforms produced at {transforms_path}'
print(f'Rig-aware dataset written to {PROCESSED_DIR}')

pixi run ns-process-data images --data /home/airlab/SousVide/gsplats/workspace/aerial_arena_20260401_165257_flat_rig/input_flat_rig --output-dir /home/airlab/SousVide/gsplats/workspace/aerial_arena_20260401_165257_flat_rig/nerfstudio_data --use-rig --rig-flat-layout --sfm-tool colmap --matching-method sequential --num-downscales 0
🌘  Running COLMAP feature extractor...0m
[18:34:08] 🎉 Done extracting COLMAP features.                                                       ]8;id=157183;file:///home/airlab/nerfstudio/nerfstudio/process_data/colmap_utils.py\colmap_utils.py]8;;\:]8;id=690011;file:///home/airlab/nerfstudio/nerfstudio/process_data/colmap_utils.py#144\144]8;;\
🚶  Running COLMAP feature matcher...0m
           🎉 Done matching COLMAP features.                                                         ]8;id=911550;file:///home/airlab/nerfstudio/nerfstudio/process_data/colmap_utils.py\colmap_utils.py]8;;\:]8;id=552551;file:///home/airlab/nerfstudio/nerfstudio/process_data

## Train Splatfacto

The run uses the rig-aware `transforms.json` from the preceding cell. `MAX_NUM_ITERATIONS` defaults to the standard 30,000-iteration Splatfacto training budget and can be adjusted for experimentation.

In [5]:
MAX_NUM_ITERATIONS = 30_000
train_cmd = [
    'pixi', 'run', 'ns-train', 'splatfacto',
    '--data', str(PROCESSED_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--max-num-iterations', str(MAX_NUM_ITERATIONS),
    '--viewer.quit-on-train-completion', 'True',
]
print(' '.join(train_cmd))
subprocess.run(train_cmd, cwd=PROJECT_ROOT, check=True)

pixi run ns-train splatfacto --data /home/airlab/SousVide/gsplats/workspace/aerial_arena_20260401_165257_flat_rig/nerfstudio_data --output-dir /home/airlab/SousVide/gsplats/workspace/outputs/aerial_arena_20260401_165257_flat_rig --max-num-iterations 30000 --viewer.quit-on-train-completion True


/home/airlab/nerfstudio/nerfstudio/field_components/activations.py:32: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @custom_fwd(cast_inputs=torch.float32)
/home/airlab/nerfstudio/nerfstudio/field_components/activations.py:39: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, g):


[18:35:02] Using --data alias for --data.pipeline.datamanager.data                                          ]8;id=808025;file:///home/airlab/nerfstudio/nerfstudio/scripts/train.py\train.py]8;;\:]8;id=869942;file:///home/airlab/nerfstudio/nerfstudio/scripts/train.py#241\241]8;;\
──────────────────────────────────────────────────────── Config ────────────────────────────────────────────────────────
TrainerConfig(
    _target=<class 'nerfstudio.engine.trainer.Trainer'>,
    output_dir=PosixPath('/home/airlab/SousVide/gsplats/workspace/outputs/aerial_arena_20260401_165257_flat_rig'),
    method_name='splatfacto',
    experiment_name=None,
    project_name='nerfstudio-project',
    timestamp='2026-07-24_183502',
    machine=MachineConfig(seed=42, num_devices=1, num_machines=1, machine_rank=0, dist_url='auto', device_type='cuda'),
    logging=LoggingConfig(
        relative_log_dir=PosixPath('.'),
        steps_per_log=10,
        max_buffer_size=20,
        local_writer=LocalWriterC

/home/airlab/nerfstudio/nerfstudio/engine/trainer.py:137: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.grad_scaler = GradScaler(enabled=self.use_grad_scaler)


╭────── viser (listening *:7007) ───────╮
│             ╷                         │
│   HTTP      │ http://localhost:7007   │
│   Websocket │ ws://localhost:7007     │
│             ╵                         │
╰───────────────────────────────────────╯
(viser) Passing ['initial_value'] as positional arguments to add_dropdown is 
deprecated. Please use keyword arguments instead: initial_value=not set
(viser) Passing ['initial_value'] as positional arguments to add_dropdown is 
deprecated. Please use keyword arguments instead: initial_value=default
(viser) Passing ['initial_value'] as positional arguments to add_dropdown is 
deprecated. Please use keyword arguments instead: initial_value=not set
(viser) Passing ['initial_value'] as positional arguments to add_dropdown is 
deprecated. Please use keyword arguments instead: initial_value=default
[NOTE] Not running eval iterations since only viewer is enabled.
Use --vis {wandb, tensorboard, viewer+wandb, viewer+tensorboard} to run with eval.


/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/site-packages/torch/_inductor/compile_fx.py:321: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


Traceback (most recent call last):
  File "/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/site-packages/gsplat/cuda/_backend.py", line 83, in 
<module>
    from gsplat import csrc as _C
ImportError: cannot import name 'csrc' from 'gsplat' 
(/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/site-packages/gsplat/__init__.py)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/subprocess.py", line 505, in run
    stdout, stderr = process.communicate(input, timeout=timeout)
  File "/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/subprocess.py", line 1141, in communicate
    stdout = self.stdout.read()
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/airlab/nerfstudio/nerfstudio/scripts/train.py", line 190, in launch
    main_func(local_rank=0, world_size=world_size,

connection handler failed
Traceback (most recent call last):
  File "/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/site-packages/viser/infra/_infra.py", line 489, in ws_handler
    await asyncio.gather(
  File "/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/site-packages/viser/infra/_infra.py", line 722, in _message_consumer
    raw = await websocket.recv()
  File "/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/site-packages/websockets/asyncio/connection.py", line 322, in recv
    raise self.protocol.close_exc from self.recv_exc
websockets.exceptions.ConnectionClosedOK: sent 1001 (going away); then received 1001 (going away)

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/site-packages/websockets/asyncio/server.py", line 376, in conn_handler
    await self.handler(connection)
  File "/home/airlab/SousVide/.pixi/envs/default/lib/python3.10/

KeyboardInterrupt: 

## Result

The trained checkpoint and configuration are under `gsplats/workspace/outputs/aerial_arena_20260401_165257_flat_rig/`. Once it completes, use the existing FiGS simulation cells from `figs_examples.ipynb` with a suitable scene/course configuration.